In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q4/checkpoints/post_cell_1_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Optimized cell 2
# Filter lineitem on GPU using vectorized comparison
lsel = lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE
flineitem = lineitem[lsel][["L_ORDERKEY"]]

# Filter orders using string literals (cuDF will cast to datetime64 on GPU)
posel = (orders.O_ORDERDATE >= "1993-08-01") & (orders.O_ORDERDATE < "1993-11-01")
forders = orders[posel]

# Use an inner merge instead of .isin for better GPU throughput
jn = forders.merge(
    flineitem,
    left_on="O_ORDERKEY",
    right_on="L_ORDERKEY",
    how="inner"
)

# Group and count on GPU
total = (
    jn.groupby("O_ORDERPRIORITY", as_index=False)
      ["O_ORDERKEY"].count()
)
